# 🎯 YOLO — Poses Corporales + Expresiones Faciales
### Dataset limpio + Entrenamiento completo

**Pasos del notebook:**
1. ✅ Montar Google Drive y verificar GPU
2. 📦 Instalar dependencias
3. 📥 Descargar datasets (COCO poses + FER2013 caras)
4. 🧹 Filtrar imágenes malas automáticamente
5. 📁 Exportar en formato YOLO
6. 🚀 Entrenar modelos
7. 💾 Guardar modelos en Google Drive

> ⚡ **Activa GPU antes de empezar:** Runtime → Change runtime type → T4 GPU

> 📂 **Los modelos finales se guardarán en:** `Mi unidad/TFM/modelos/`

In [ ]:
# ─────────────────────────────────────────
# CELDA 1 — Montar Google Drive + Verificar GPU
# ─────────────────────────────────────────
import torch
import shutil
import os
from google.colab import drive

# ── Montar Drive PRIMERO ──
print('📂 Montando Google Drive...')
drive.mount('/content/drive')

# ── Crear carpetas en Drive ──
DRIVE_BASE = '/content/drive/MyDrive/TFM'
DRIVE_MODELOS   = f'{DRIVE_BASE}/modelos'
DRIVE_RUNS      = f'{DRIVE_BASE}/runs'
DRIVE_CHECKPOINTS = f'{DRIVE_BASE}/checkpoints'

for carpeta in [DRIVE_MODELOS, DRIVE_RUNS, DRIVE_CHECKPOINTS]:
    os.makedirs(carpeta, exist_ok=True)

print(f'✅ Google Drive montado')
print(f'   Carpeta base  : {DRIVE_BASE}')
print(f'   Modelos finales: {DRIVE_MODELOS}')

# ── Verificar GPU ──
print()
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU disponible: {gpu}')
    print(f'   Memoria GPU: {mem:.1f} GB')
else:
    print('⚠️  No se detectó GPU.')
    print('   Ve a: Runtime → Change runtime type → T4 GPU')

# ── Espacio disponible ──
total, used, free = shutil.disk_usage('/content')
print(f'\n💾 Espacio libre en Colab : {free // (2**30)} GB')

total_d, used_d, free_d = shutil.disk_usage('/content/drive/MyDrive')
print(f'💾 Espacio libre en Drive : {free_d // (2**30)} GB')

In [ ]:
# ─────────────────────────────────────────
# CELDA 2 — Instalar dependencias
# ─────────────────────────────────────────
print('📦 Instalando dependencias...')
!pip install -q fiftyone ultralytics opencv-python tqdm
print('✅ Dependencias instaladas')

In [ ]:
# ─────────────────────────────────────────
# CELDA 3 — Configuración general
# ─────────────────────────────────────────

CONFIG = {
    # Cantidad de imágenes a descargar
    'max_samples'     : 6000,

    # Filtros de calidad
    'min_keypoints'   : 8,
    'min_person_area' : 0.05,

    # División train/val
    'val_split'       : 0.2,

    # Rutas — todo en /content (temporal, rápido)
    'output_dir'      : '/content/dataset_final',
    'image_size'      : 640,

    # Entrenamiento
    'epochs_poses'    : 50,
    'epochs_faces'    : 30,

    # Google Drive — destino final
    'drive_modelos'   : DRIVE_MODELOS,
    'drive_runs'      : DRIVE_RUNS,
    'drive_checkpoints': DRIVE_CHECKPOINTS,
}

print('✅ Configuración lista')
print(f"   Muestras a descargar : {CONFIG['max_samples']}")
print(f"   Min keypoints        : {CONFIG['min_keypoints']}")
print(f"   Split val            : {CONFIG['val_split']*100:.0f}%")
print(f"   Modelos → Drive      : {CONFIG['drive_modelos']}")

In [ ]:
# ─────────────────────────────────────────
# CELDA 4 — Descargar COCO 2017 (poses)
# ─────────────────────────────────────────
import fiftyone.zoo as foz
import fiftyone as fo

print('📥 Descargando COCO-2017 (solo personas con keypoints)...')
print('   Esto toma ~5-10 minutos en Colab\n')

dataset_poses = foz.load_zoo_dataset(
    'coco-2017',
    split='train',
    label_types=['detections', 'keypoints'],
    classes=['person'],
    max_samples=CONFIG['max_samples'],
    dataset_name='coco_poses_raw',
    overwrite=True,
)

print(f'\n✅ COCO descargado: {len(dataset_poses)} imágenes')

In [ ]:
# ─────────────────────────────────────────
# CELDA 5 — Descargar FER2013 (expresiones)
# ─────────────────────────────────────────
import os

print('📥 Descargando FER2013 (expresiones faciales)...')

# OPCIÓN A: Si tienes kaggle.json (recomendado)
kaggle_json = '/content/kaggle.json'

if os.path.exists(kaggle_json):
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp /content/kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d msambare/fer2013 -p /content/fer2013_raw --unzip
    print('✅ FER2013 descargado con Kaggle API')
    usar_fer_manual = True
else:
    print('⚠️  No se encontró kaggle.json')
    print('   Intentando descarga directa...')
    try:
        !wget -q -O /content/fer2013.zip "https://www.kaggle.com/api/v1/datasets/download/msambare/fer2013"
        !unzip -q /content/fer2013.zip -d /content/fer2013_raw/
        print('✅ FER2013 descargado')
        usar_fer_manual = True
    except:
        print('   Para usar FER2013:')
        print('   1. Ve a kaggle.com → Account → Create API Token')
        print('   2. Sube el kaggle.json a /content/ en Colab')
        print('   3. Vuelve a correr esta celda')
        usar_fer_manual = False

In [ ]:
# ─────────────────────────────────────────
# CELDA 6 — Filtrar imágenes de poses
# ─────────────────────────────────────────
import numpy as np
from tqdm import tqdm

print('🧹 Filtrando imágenes de poses...')
print(f'   Criterios:')
print(f'   • Mínimo {CONFIG["min_keypoints"]} keypoints visibles')
print(f'   • Persona ocupa al menos {CONFIG["min_person_area"]*100:.0f}% del frame')
print()

muestras_validas = []

for muestra in tqdm(dataset_poses, desc='Evaluando calidad'):
    if muestra.keypoints is None or len(muestra.keypoints.keypoints) == 0:
        continue

    img_w = muestra.metadata.width  if muestra.metadata else 640
    img_h = muestra.metadata.height if muestra.metadata else 480
    img_area = img_w * img_h

    persona_valida = False
    for kp in muestra.keypoints.keypoints:
        if kp.label != 'person':
            continue
        puntos   = kp.points
        visibles = [p for p in puntos if p[0] > 0 and p[1] > 0]

        if len(visibles) < CONFIG['min_keypoints']:
            continue

        xs = [p[0] for p in visibles]
        ys = [p[1] for p in visibles]
        area_persona = ((max(xs)-min(xs)) * img_w * (max(ys)-min(ys)) * img_h) / img_area

        if area_persona >= CONFIG['min_person_area']:
            persona_valida = True
            break

    if persona_valida:
        muestras_validas.append(muestra)

descartadas = len(dataset_poses) - len(muestras_validas)
print(f'\n📊 Resultados del filtrado:')
print(f'   Originales    : {len(dataset_poses):,}')
print(f'   ✅ Válidas    : {len(muestras_validas):,}')
print(f'   ❌ Descartadas: {descartadas:,} (comida, animales, personas pequeñas)')

In [ ]:
# ─────────────────────────────────────────
# CELDA 7 — Exportar poses a formato YOLO
# ─────────────────────────────────────────
import cv2
from pathlib import Path

output_base = CONFIG['output_dir']

for split in ['train', 'val']:
    Path(f'{output_base}/poses/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{output_base}/poses/labels/{split}').mkdir(parents=True, exist_ok=True)

np.random.shuffle(muestras_validas)
n_val  = int(len(muestras_validas) * CONFIG['val_split'])
splits = {'val': muestras_validas[:n_val], 'train': muestras_validas[n_val:]}

print('📁 Exportando poses al formato YOLO...')
for split_name, split_muestras in splits.items():
    print(f'\n   {split_name}: {len(split_muestras)} imágenes')
    for muestra in tqdm(split_muestras, desc=f'   {split_name}'):
        img_path = muestra.filepath
        nombre   = Path(img_path).stem
        img      = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (CONFIG['image_size'], CONFIG['image_size']))
        cv2.imwrite(f'{output_base}/poses/images/{split_name}/{nombre}.jpg', img)

        lineas = []
        if muestra.keypoints:
            for kp in muestra.keypoints.keypoints:
                if kp.label != 'person':
                    continue
                puntos = kp.points
                xs = [p[0] for p in puntos if p[0] > 0]
                ys = [p[1] for p in puntos if p[1] > 0]
                if not xs or not ys:
                    continue
                cx = (min(xs)+max(xs))/2
                cy = (min(ys)+max(ys))/2
                bw = max(xs)-min(xs)
                bh = max(ys)-min(ys)
                kp_str = ''.join([f' {p[0]:.6f} {p[1]:.6f} {2 if p[0]>0 and p[1]>0 else 0}' for p in puntos])
                lineas.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}{kp_str}')

        with open(f'{output_base}/poses/labels/{split_name}/{nombre}.txt', 'w') as f:
            f.write('\n'.join(lineas))

# YAML para YOLOv8/v11
yaml_poses = f"""path: {output_base}/poses
train: images/train
val:   images/val
kpt_shape: [17, 3]
names:\n  0: person
"""
with open(f'{output_base}/poses/poses.yaml', 'w') as f:
    f.write(yaml_poses)

print(f'\n✅ Poses exportadas → {output_base}/poses/')

In [ ]:
# ─────────────────────────────────────────
# CELDA 8 — Exportar expresiones a formato YOLO
# ─────────────────────────────────────────

EXPRESIONES = {
    'angry': 0, 'disgust': 1, 'fear': 2,
    'happy': 3, 'sad': 4, 'surprise': 5, 'neutral': 6
}

for split in ['train', 'val']:
    Path(f'{output_base}/faces/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{output_base}/faces/labels/{split}').mkdir(parents=True, exist_ok=True)

fer_path = Path('/content/fer2013_raw/train')
if fer_path.exists():
    print('📁 Exportando expresiones faciales al formato YOLO...')
    for expresion, idx in EXPRESIONES.items():
        src_dir  = fer_path / expresion
        if not src_dir.exists():
            continue
        archivos = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
        np.random.shuffle(archivos)
        n_val_f  = int(len(archivos) * CONFIG['val_split'])
        grupos   = {'val': archivos[:n_val_f], 'train': archivos[n_val_f:]}

        for split_name, imgs in grupos.items():
            for img_path in tqdm(imgs, desc=f'   {expresion}/{split_name}'):
                nombre = f'{expresion}_{img_path.stem}'
                img    = cv2.imread(str(img_path))
                if img is None:
                    continue
                img = cv2.resize(img, (CONFIG['image_size'], CONFIG['image_size']))
                cv2.imwrite(f'{output_base}/faces/images/{split_name}/{nombre}.jpg', img)
                with open(f'{output_base}/faces/labels/{split_name}/{nombre}.txt', 'w') as f:
                    f.write(f'{idx} 0.5 0.5 1.0 1.0\n')
else:
    print('⚠️  FER2013 no encontrado — sube kaggle.json y corre la celda 5 nuevamente')

# YAML para YOLOv8/v11
yaml_faces = f"""path: {output_base}/faces
train: images/train
val:   images/val
nc: 7
names:\n  0: angry\n  1: disgust\n  2: fear\n  3: happy\n  4: sad\n  5: surprise\n  6: neutral
"""
with open(f'{output_base}/faces/faces.yaml', 'w') as f:
    f.write(yaml_faces)

print(f'\n✅ Expresiones exportadas → {output_base}/faces/')

In [ ]:
# ─────────────────────────────────────────
# CELDA 9 — Resumen del dataset
# ─────────────────────────────────────────
from pathlib import Path

print('='*50)
print('  RESUMEN DATASET FINAL')
print('='*50)

total = 0
for tarea in ['poses', 'faces']:
    for split in ['train', 'val']:
        ruta = Path(f'{output_base}/{tarea}/images/{split}')
        if ruta.exists():
            n = len(list(ruta.glob('*.jpg')))
            total += n
            print(f'  {tarea:8} / {split:5} : {n:6,} imágenes')

print(f'\n  TOTAL: {total:,} imágenes limpias y listas')
print('='*50)

# Verificar espacio antes de entrenar
total_c, used_c, free_c = shutil.disk_usage('/content')
print(f'\n💾 Espacio libre restante en Colab: {free_c // (2**30)} GB')
if free_c // (2**30) < 5:
    print('⚠️  ADVERTENCIA: menos de 5 GB libres — considera reducir max_samples')
else:
    print('✅ Espacio suficiente para entrenar')

In [ ]:
# ─────────────────────────────────────────
# CELDA 10 — Entrenar modelo de POSES
# ─────────────────────────────────────────
from ultralytics import YOLO

print('🚀 Entrenando YOLOv8 — Detección de Poses...')
print(f'   Épocas : {CONFIG["epochs_poses"]}')
print(f'   Imagen : {CONFIG["image_size"]}px')
print()

model_poses = YOLO('yolov8n-pose.pt')

results_poses = model_poses.train(
    data    = f'{output_base}/poses/poses.yaml',
    epochs  = CONFIG['epochs_poses'],
    imgsz   = CONFIG['image_size'],
    batch   = 16,
    device  = 0,
    project = '/content/runs',
    name    = 'poses',
    exist_ok= True,
)

# ── Guardar checkpoint en Drive inmediatamente ──
print('\n💾 Guardando modelo de poses en Drive...')
shutil.copy(
    '/content/runs/poses/weights/best.pt',
    f"{CONFIG['drive_checkpoints']}/yolo_poses_best.pt"
)
print(f"   ✅ Guardado en Drive: TFM/checkpoints/yolo_poses_best.pt")
print('\n✅ Entrenamiento de poses completado')

In [ ]:
# ─────────────────────────────────────────
# CELDA 11 — Entrenar modelo de EXPRESIONES
# ─────────────────────────────────────────
print('🚀 Entrenando YOLOv8 — Expresiones Faciales...')

model_faces = YOLO('yolov8n-cls.pt')

results_faces = model_faces.train(
    data    = f'{output_base}/faces',
    epochs  = CONFIG['epochs_faces'],
    imgsz   = CONFIG['image_size'],
    batch   = 32,
    device  = 0,
    project = '/content/runs',
    name    = 'faces',
    exist_ok= True,
)

# ── Guardar checkpoint en Drive inmediatamente ──
print('\n💾 Guardando modelo de expresiones en Drive...')
shutil.copy(
    '/content/runs/faces/weights/best.pt',
    f"{CONFIG['drive_checkpoints']}/yolo_faces_best.pt"
)
print(f"   ✅ Guardado en Drive: TFM/checkpoints/yolo_faces_best.pt")
print('\n✅ Entrenamiento de expresiones completado')

In [ ]:
# ─────────────────────────────────────────
# CELDA 12 — Empaquetar y guardar en Drive
# ─────────────────────────────────────────
import shutil
import os

print('💾 Empaquetando modelos finales...')

# ── Copiar modelos finales a Drive/TFM/modelos ──
shutil.copy(
    '/content/runs/poses/weights/best.pt',
    f"{CONFIG['drive_modelos']}/yolo_poses.pt"
)
shutil.copy(
    '/content/runs/faces/weights/best.pt',
    f"{CONFIG['drive_modelos']}/yolo_faces.pt"
)

# ── Crear zip en Drive directamente ──
zip_path = f"{CONFIG['drive_modelos']}/modelos_yolo"
shutil.make_archive(zip_path, 'zip', CONFIG['drive_modelos'])

size_mb = os.path.getsize(f'{zip_path}.zip') / 1e6

print(f'\n📦 modelos_yolo.zip ({size_mb:.1f} MB)')
print(f"   Guardado en Drive: TFM/modelos/modelos_yolo.zip")
print(f"   También disponibles individualmente:")
print(f"     • TFM/modelos/yolo_poses.pt")
print(f"     • TFM/modelos/yolo_faces.pt")

# ── Copiar logs de entrenamiento ──
print('\n📊 Guardando logs de entrenamiento...')
for run in ['poses', 'faces']:
    src = f'/content/runs/{run}'
    dst = f"{CONFIG['drive_runs']}/{run}"
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'   ✅ Logs {run} → Drive/TFM/runs/{run}/')

print('\n✅ ¡Todo guardado en Google Drive!')
print(f'\n📂 Estructura en tu Drive:')
print(f'   MyDrive/TFM/')
print(f'   ├── modelos/')
print(f'   │   ├── yolo_poses.pt')
print(f'   │   ├── yolo_faces.pt')
print(f'   │   └── modelos_yolo.zip')
print(f'   ├── checkpoints/')
print(f'   │   ├── yolo_poses_best.pt')
print(f'   │   └── yolo_faces_best.pt')
print(f'   └── runs/')
print(f'       ├── poses/   (métricas y gráficas)')
print(f'       └── faces/   (métricas y gráficas)')

In [ ]:
# ─────────────────────────────────────────
# CELDA 13 — (Opcional) Descargar a tu PC
# ─────────────────────────────────────────
# Los modelos ya están en Drive. Corre esto SOLO si
# además quieres una copia directa en tu computador.

from google.colab import files

print('⬇️  Descargando modelos_yolo.zip a tu computador...')
print('   (El archivo también está en Drive/TFM/modelos/)')
print()

files.download(f"{CONFIG['drive_modelos']}/modelos_yolo.zip")

print('\n✅ Descarga iniciada en tu navegador')
print('   El archivo irá a tu carpeta de Descargas.')
print('   Desde ahí puedes moverlo a /Volumes/Echo 13 SSD/TFM/images/')

## ✅ ¿Qué hacer con los modelos descargados?

Los modelos quedan en `MyDrive/TFM/modelos/`. Para pasarlos a tu SSD, abre Terminal en tu Mac y ejecuta:

```bash
cp ~/Library/CloudStorage/GoogleDrive-tucorreo@gmail.com/Mi\ unidad/TFM/modelos/yolo_poses.pt \
   "/Volumes/Echo 13 SSD/TFM/images/"

cp ~/Library/CloudStorage/GoogleDrive-tucorreo@gmail.com/Mi\ unidad/TFM/modelos/yolo_faces.pt \
   "/Volumes/Echo 13 SSD/TFM/images/"
```

